# Capstone Modeling Notebook

**Project:** County-level traffic fatality modeling  
**Research question:** Which county-level crash, socioeconomic, and risk factors are associated with higher traffic fatality rates?  
**Target variable:** `fatalities_per_100k`

This notebook starts fresh from the merged county-level workflow and will be extended through the six modeling weeks:

1. Polynomial and interaction terms
2. Ridge, lasso, and elastic net regression
3. Forward/backward selection, PCR, and PLSR
4. Logistic regression and feature scaling
5. Support vector machines
6. Decision trees and random forests

## 1. Setup

Run this cell first. It imports the libraries used throughout the notebook and sets a consistent random seed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV, LogisticRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

RANDOM_STATE = 42
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 2. Load Raw Data

Update `DATA_DIR` if your CSV files are stored somewhere different. This notebook expects the same source files from the previous milestone workflow.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/workspaces/modB2-Week7-milestone')

accident_df = pd.read_csv(DATA_DIR / 'accident.csv')
risk_df = pd.read_csv(DATA_DIR / 'National_Risk_Index_Counties.csv')
population_df = pd.read_csv(DATA_DIR / 'PopulationEstimates.csv', encoding='latin-1')
poverty_df = pd.read_csv(DATA_DIR / 'Poverty2023.csv')

print('Accident:', accident_df.shape)
print('Risk:', risk_df.shape)
print('Population:', population_df.shape)
print('Poverty:', poverty_df.shape)

## 3. Build the County-Level Modeling Dataset

This section recreates the cleaned county-level dataset used in the previous milestone. It keeps the target variable and the reduced feature set that removed highly redundant crash variables.

In [ ]:
# Population: keep 2022 county population estimates
population_clean = population_df[population_df['Attribute'] == 'POP_ESTIMATE_2022'].copy()
population_clean = population_clean.rename(columns={'FIPStxt': 'FIPS', 'Value': 'population'})
population_clean['FIPS'] = population_clean['FIPS'].astype(str).str.zfill(5)
population_clean = population_clean[
    (population_clean['FIPS'].str.len() == 5) &
    (population_clean['FIPS'].str[2:] != '000')
].copy()

# Poverty: keep 2023 all-ages poverty rate
poverty_clean = poverty_df[poverty_df['Attribute'] == 'PCTPOVALL_2023'].copy()
poverty_clean = poverty_clean.rename(columns={'FIPS_Code': 'FIPS', 'Value': 'poverty_rate'})
poverty_clean['FIPS'] = poverty_clean['FIPS'].astype(str).str.zfill(5)
poverty_clean = poverty_clean[
    (poverty_clean['FIPS'].str.len() == 5) &
    (poverty_clean['FIPS'].str[2:] != '000')
].copy()

# FEMA National Risk Index: keep summary scores
risk_clean = risk_df[['STCOFIPS', 'RISK_SCORE', 'SOVI_SCORE', 'RESL_SCORE']].copy()
risk_clean = risk_clean.rename(columns={'STCOFIPS': 'FIPS'})
risk_clean['FIPS'] = risk_clean['FIPS'].astype(str).str.zfill(5)

print('Population clean:', population_clean.shape)
print('Poverty clean:', poverty_clean.shape)
print('Risk clean:', risk_clean.shape)

In [ ]:
# Create county FIPS in FARS accident data
accident_df['STATE'] = accident_df['STATE'].astype(str).str.zfill(2)
accident_df['COUNTY'] = accident_df['COUNTY'].astype(str).str.zfill(3)
accident_df['FIPS'] = accident_df['STATE'] + accident_df['COUNTY']

# Aggregate crash data to county level
county_df = (
    accident_df
    .groupby('FIPS', as_index=False)
    .agg({
        'FATALS': 'sum',
        'PERSONS': 'mean',
        'VE_TOTAL': 'mean',
        'PEDS': 'mean',
        'PERNOTMVIT': 'mean',
        'PERMVIT': 'mean'
    })
)

print('County crash data:', county_df.shape)
display(county_df.head())

In [ ]:
# Merge county-level datasets
df = county_df.merge(population_clean, on='FIPS', how='inner')
df = df.merge(poverty_clean, on='FIPS', how='left')
df = df.merge(risk_clean, on='FIPS', how='left')

# Create target variable
df['fatalities_per_100k'] = (df['FATALS'] / df['population']) * 100000
df = df.replace([np.inf, -np.inf], np.nan)

# Remove columns that are not needed for modeling, if present
df = df.drop(columns=['State', 'Area_Name_x', 'Area_Name_y', 'Attribute_x', 'Attribute_y', 'Stabr'], errors='ignore')

print('Merged dataset:', df.shape)
display(df.head())
print(df['fatalities_per_100k'].describe())

## 4. Final Feature Set and Train/Test Split

The reduced feature set removes `PERMVIT` and `PERNOTMVIT` because earlier correlation and PCA work showed that they were highly redundant with `PERSONS` and `PEDS`.

The same 80/20 train/test split will be reused for all models so that performance comparisons are consistent.

In [ ]:
# Filter small counties and extreme fatality-rate outliers
df_filtered = df[
    (df['population'] > 1000) &
    (df['fatalities_per_100k'] < 200)
].copy()

model_cols_clean = [
    'PERSONS',
    'VE_TOTAL',
    'PEDS',
    'poverty_rate',
    'RISK_SCORE',
    'SOVI_SCORE',
    'RESL_SCORE'
]

df_model = df_filtered[model_cols_clean + ['fatalities_per_100k']].dropna().copy()

X = df_model[model_cols_clean]
y = df_model['fatalities_per_100k']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print('Filtered dataset:', df_filtered.shape)
print('Modeling dataset:', df_model.shape)
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
display(df_model.head())

## 5. Modeling Helper Functions

These functions keep the model comparisons consistent across all six weeks.

In [ ]:
results = []

def regression_metrics(model_name, model, X_train, X_test, y_train, y_test, notes=''):
    """Fit a model, evaluate it, and store results in the master results table."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cv_scores = cross_val_score(model, X_train, y_train, cv=CV, scoring='r2')
    
    row = {
        'model': model_name,
        'test_r2': r2_score(y_test, y_pred),
        'test_rmse': mean_squared_error(y_test, y_pred, squared=False),
        'test_mae': mean_absolute_error(y_test, y_pred),
        'cv_r2_mean': cv_scores.mean(),
        'cv_r2_std': cv_scores.std(),
        'notes': notes
    }
    results.append(row)
    return row, y_pred

def show_results():
    """Display the master model comparison table."""
    return pd.DataFrame(results).sort_values('test_rmse')

def plot_actual_vs_predicted(y_test, y_pred, title):
    plt.figure()
    plt.scatter(y_test, y_pred, alpha=0.7)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val])
    plt.xlabel('Actual fatalities per 100k')
    plt.ylabel('Predicted fatalities per 100k')
    plt.title(title)
    plt.show()

def plot_residuals(y_test, y_pred, title):
    residuals = y_test - y_pred
    plt.figure()
    plt.scatter(y_pred, residuals, alpha=0.7)
    plt.axhline(0)
    plt.xlabel('Predicted fatalities per 100k')
    plt.ylabel('Residuals')
    plt.title(title)
    plt.show()

# Week 1: Linear Regression Part 1

This section establishes the baseline regression model and then tests whether polynomial and interaction terms improve performance.

In [ ]:
# Baseline multiple linear regression
linear_model = LinearRegression()
linear_row, linear_pred = regression_metrics(
    'Baseline Linear Regression',
    linear_model,
    X_train, X_test, y_train, y_test,
    notes='Unscaled baseline using reduced feature set'
)

linear_row

In [ ]:
plot_actual_vs_predicted(y_test, linear_pred, 'Baseline Linear Regression: Actual vs. Predicted')
plot_residuals(y_test, linear_pred, 'Baseline Linear Regression: Residual Plot')

In [ ]:
# Coefficients from the baseline linear regression
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': linear_model.coef_
}).sort_values('coefficient', ascending=False)

display(coef_df)

In [ ]:
# Polynomial terms, degree 2
poly_model = Pipeline(steps=[
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

poly_row, poly_pred = regression_metrics(
    'Polynomial Regression Degree 2',
    poly_model,
    X_train, X_test, y_train, y_test,
    notes='Degree-2 polynomial and interaction terms with scaling'
)

poly_row

In [ ]:
plot_actual_vs_predicted(y_test, poly_pred, 'Polynomial Regression: Actual vs. Predicted')
plot_residuals(y_test, poly_pred, 'Polynomial Regression: Residual Plot')

In [ ]:
show_results()

## Week 1 Notes

Use this space to write your interpretation after running the models. Focus on whether polynomial and interaction terms improved predictive performance, whether the model appears to overfit, and what the residual plots suggest.

In [ ]:
# Add written notes here after reviewing the model outputs.

# Week 2: Ridge, Lasso, and Elastic Net Regression

This section will be completed next.

In [ ]:
# Week 2 placeholder
# RidgeCV, LassoCV, and ElasticNetCV will go here.

# Week 3: Feature Selection, PCR, and PLSR

This section will be completed later.

In [ ]:
# Week 3 placeholder

# Week 4: Logistic Regression and Feature Scaling

Logistic regression requires a classification target. Later, we will create a binary target such as high-fatality county vs. lower-fatality county.

In [ ]:
# Week 4 placeholder
# Example later: df_model['high_fatality_county'] = ...

# Week 5: Support Vector Machines

This section will be completed later.

In [ ]:
# Week 5 placeholder

# Week 6: Decision Trees and Random Forests

This section will revisit and cleanly reproduce the random forest analysis from the previous milestone.

In [ ]:
# Week 6 placeholder

# Final Model Comparison

This table will grow as each week's models are added.

In [ ]:
show_results()